# Map Arvoredo dataset using Follium

In [3]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import json
from folium import GeoJson
import folium

In [4]:
# load the CSV file into a DataFrame
csv_path = "../data/arvoredo_cleaned.csv"
df = pd.read_csv(csv_path)
df.head()

,Código SIG Novo,Morada,Espécie,PAP,Manutenção,Ocupação,Local,Tipologia,Freguesia,Nome Vulgar,longitude,latitude
0,1,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095224,38.774517
1,2,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095219,38.774579
2,3,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095178,38.774387
3,4,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095173,38.774452
4,5,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095168,38.774515


In [6]:
# Optimize data
# Keep only essential columns
df_minimal = df[['latitude', 'longitude', 'Nome Vulgar', 'Espécie', 'Local']].copy()

# Round coordinates to 4 decimals (still city-accurate)
df_minimal['latitude'] = df_minimal['latitude'].round(4)
df_minimal['longitude'] = df_minimal['longitude'].round(4)

In [ ]:
from folium.plugins import FastMarkerCluster

map_center = [df_minimal["latitude"].mean(), df_minimal["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles="Esri.WorldImagery")

coords = df_minimal.loc[
    df_minimal["latitude"].notna() & df_minimal["longitude"].notna(),
    ["latitude", "longitude"]
].values.tolist()

FastMarkerCluster(coords, 
                  name="Trees",
                  icon_create_function=icon_create_function,
).add_to(m)

In [ ]:
from folium.plugins import FastMarkerCluster

map_center = [df_minimal["latitude"].mean(), df_minimal["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")

coords = df_minimal[["latitude", "longitude", "Espécie"]].values.tolist()

callback = """
function (row) {

    var icon = L.AwesomeMarkers.icon({
        icon: 'leaf',
        prefix: 'fa',
        markerColor: 'green'
    });

    var marker = L.marker(
        [row[0], row[1]],
        {icon: icon}
    );

    marker.bindPopup(
        '<b>Species:</b> ' + row[2]
    );

    return marker;
}
"""

icon_create_function = """
function(cluster) {
    var childCount = cluster.getChildCount();
    var color = '#1b5e20';

    if (childCount < 5) {
        color = '#e8f5e9';
    } else if (childCount < 20) {
        color = '#c8e6c9';
    } else if (childCount < 50) {
        color = '#66bb6a';
    }

    return new L.DivIcon({
        html: '<div style="background-color:' + color + '; width: 40px; height: 40px; border-radius: 20px; display: flex; align-items: center; justify-content: center; color: white; font-weight: bold;">' + childCount + '</div>',
        className: 'marker-cluster',
        iconSize: new L.Point(40, 40)
    });
}
"""

fast_cluster = FastMarkerCluster(
    coords,
    name="Lisbon Trees",
    callback=callback,
    icon_create_function=icon_create_function,
)

fast_cluster.add_to(m)

# Additional basemaps
folium.TileLayer(
    "CartoDB Positron",
    name="Light Map"
).add_to(m)

folium.TileLayer(
    "CartoDB Voyager",
    name="Voyager"
).add_to(m)

folium.TileLayer(
    "OpenTopoMap",
    name="Topographic"
).add_to(m)

folium.TileLayer(
    "Esri.WorldImagery",
    name="Satellite"
).add_to(m)

folium.LayerControl().add_to(m)
m

AssertionError: 

In [ ]:
m.save('../data/arvoredo_map_layers.html')
print("Map saved to '../data/arvoredo_map_layers.html'")